In [4]:
import os

print("Exists:", os.path.exists(r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\train_images"))
print("Exists:", os.path.exists(r"C:\Users\HP\Desktop\OPENLAB_PROJECT\data"))
print("Exists:", os.path.exists(r"C:\Users\HP\Desktop\OPENLAB_PROJECT"))

Exists: True
Exists: True
Exists: True


In [6]:
import cv2
import numpy as np
import os
import random
from tqdm import tqdm

# ==========================
# PATHS
# ==========================
REAL_TRAIN_DIR = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\train_images"
REAL_VAL_DIR   = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\val_images"

FAKE_TRAIN_DIR = r"C:\Users\HP\Desktop\OPENLAB_PROJECT\data\ai_train_fake"
FAKE_VAL_DIR   = r"C:\Users\HP\Desktop\OPENLAB_PROJECT\data\ai_val_fake"

os.makedirs(FAKE_TRAIN_DIR, exist_ok=True)
os.makedirs(FAKE_VAL_DIR, exist_ok=True)

# ==========================
# MANIPULATION FUNCTIONS
# ==========================

def random_scratch_overlay(image):
    h, w = image.shape[:2]
    for _ in range(random.randint(3, 8)):
        x1 = random.randint(0, w)
        y1 = random.randint(0, h)
        x2 = random.randint(0, w)
        y2 = random.randint(0, h)
        thickness = random.randint(1, 3)
        color = (random.randint(180,255), random.randint(180,255), random.randint(180,255))
        cv2.line(image, (x1, y1), (x2, y2), color, thickness)
    return image

def exaggerate_contrast(image):
    alpha = random.uniform(1.2, 1.6)  # contrast
    beta = random.randint(-20, 20)    # brightness
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

def local_blur(image):
    h, w = image.shape[:2]
    x = random.randint(0, w-100)
    y = random.randint(0, h-100)
    patch = image[y:y+100, x:x+100]
    patch = cv2.GaussianBlur(patch, (15,15), 0)
    image[y:y+100, x:x+100] = patch
    return image

def slight_distortion(image):
    h, w = image.shape[:2]
    pts1 = np.float32([[0,0],[w,0],[0,h]])
    shift = 10
    pts2 = np.float32([
        [random.randint(0,shift), random.randint(0,shift)],
        [w-random.randint(0,shift), random.randint(0,shift)],
        [random.randint(0,shift), h-random.randint(0,shift)]
    ])
    M = cv2.getAffineTransform(pts1, pts2)
    return cv2.warpAffine(image, M, (w,h))

def manipulate_image(image):
    if random.random() > 0.5:
        image = exaggerate_contrast(image)
    if random.random() > 0.5:
        image = random_scratch_overlay(image)
    if random.random() > 0.5:
        image = local_blur(image)
    if random.random() > 0.5:
        image = slight_distortion(image)
    return image

# ==========================
# GENERATION FUNCTION
# ==========================

def generate_folder(input_dir, output_dir, max_images):
    image_list = os.listdir(input_dir)
    random.shuffle(image_list)
    image_list = image_list[:max_images]

    for img_name in tqdm(image_list):
        try:
            img_path = os.path.join(input_dir, img_name)
            image = cv2.imread(img_path)

            fake_image = manipulate_image(image)

            save_path = os.path.join(output_dir, img_name)
            cv2.imwrite(save_path, fake_image)

        except Exception as e:
            print("Error:", img_name, e)

# ==========================
# RUN GENERATION
# ==========================

print("Generating TRAIN fake images...")
generate_folder(REAL_TRAIN_DIR, FAKE_TRAIN_DIR, max_images=3000)

print("Generating VAL fake images...")
generate_folder(REAL_VAL_DIR, FAKE_VAL_DIR, max_images=800)

print("FAST AI Dataset Generation Completed.")

Generating TRAIN fake images...


100%|██████████| 3000/3000 [00:39<00:00, 76.14it/s]


Generating VAL fake images...


100%|██████████| 800/800 [00:10<00:00, 75.32it/s]

FAST AI Dataset Generation Completed.


In [7]:
import os
import subprocess

path = r"C:\Users\HP\Desktop\OPENLAB_PROJECT\data\ai_train_fake"
subprocess.run(["explorer", path])

CompletedProcess(args=['explorer', 'C:\\Users\\HP\\Desktop\\OPENLAB_PROJECT\\data\\ai_train_fake'], returncode=1)